<a href="https://colab.research.google.com/github/padhikari26/Care100/blob/main/TCS_COVID_19_Research_Project_Final_PurnimaAdhikari.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ====================================
# Step 1: Imports
# ====================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr, ttest_ind
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.integrate import odeint

sns.set(style="whitegrid", palette="muted")

# ====================================
# Step 2: Load & Prepare Data
# ====================================
owid = pd.read_csv('https://covid.ourworldindata.org/data/owid-covid-data.csv', parse_dates=['date'])
ghsi = pd.read_csv('ghsi.csv')  # must be pre-downloaded

ghsi.rename(columns={'Country': 'location', 'Overall Score': 'ghsi_score'}, inplace=True)

owid_data = owid[['location', 'date', 'total_cases_per_million',
                  'total_deaths_per_million', 'people_vaccinated_per_hundred',
                  'stringency_index']]

snapshot_date = '2021-12-31'
snapshot = owid_data[owid_data['date'] == pd.to_datetime(snapshot_date)].dropna()
summary = snapshot.merge(ghsi[['location', 'ghsi_score']], on='location', how='left').dropna()

# ====================================
# Step 3: EDA
# ====================================
sns.pairplot(summary[['total_cases_per_million', 'total_deaths_per_million',
                      'people_vaccinated_per_hundred', 'stringency_index', 'ghsi_score']])
plt.suptitle("Pairwise Relationships", y=1.02)
plt.show()

plt.figure(figsize=(6, 5))
sns.heatmap(summary.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

# ====================================
# Step 4: Statistical Tests
# ====================================
print("Pearson:", pearsonr(summary['people_vaccinated_per_hundred'], summary['total_deaths_per_million']))
print("Spearman:", spearmanr(summary['people_vaccinated_per_hundred'], summary['total_deaths_per_million']))

median_vax = summary['people_vaccinated_per_hundred'].median()
t_stat, p_val = ttest_ind(
    summary[summary['people_vaccinated_per_hundred'] > median_vax]['total_deaths_per_million'],
    summary[summary['people_vaccinated_per_hundred'] <= median_vax]['total_deaths_per_million']
)
print(f"T-Test: t={t_stat:.2f}, p={p_val:.5f}")

# ====================================
# Step 5: ML Models – Linear Regression & KNN
# ====================================
X = summary[['people_vaccinated_per_hundred', 'stringency_index', 'ghsi_score']]
y = summary['total_deaths_per_million']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("\nLinear Regression:")
print("R²:", r2_score(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))

# K-Nearest Neighbors Regression
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
print("\nKNN Regression:")
print("R²:", r2_score(y_test, y_pred_knn))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_knn)))

# Compare visually
plt.figure(figsize=(10,5))
plt.plot(y_test.values, label='True')
plt.plot(y_pred_lr, label='Predicted (Linear)', linestyle='--')
plt.plot(y_pred_knn, label='Predicted (KNN)', linestyle='--')
plt.legend()
plt.title("Prediction Comparison")
plt.ylabel("Deaths per Million")
plt.show()

# ====================================
# Step 6: Experimental Curiosity
# ====================================
plt.figure(figsize=(10,6))
sns.scatterplot(data=summary, x='ghsi_score', y='total_deaths_per_million',
                hue='people_vaccinated_per_hundred', palette='viridis')
plt.title("Does Higher GHSI → Lower Deaths?")
plt.show()

# ====================================
# Step 7: Time Series Trends
# ====================================
countries = ['United States', 'India', 'Brazil', 'Germany', 'South Africa']
ts = owid_data[owid_data['location'].isin(countries)].set_index('date')

plt.figure(figsize=(14, 5))
for c in countries:
    plt.plot(ts[ts['location']==c]['people_vaccinated_per_hundred'], label=c)
plt.title("Vaccination Uptake Over Time")
plt.legend()
plt.show()

plt.figure(figsize=(14, 5))
for c in countries:
    plt.plot(ts[ts['location']==c]['stringency_index'], label=c)
plt.title("Policy Stringency Over Time")
plt.legend()
plt.show()

# ====================================
# Step 8: Epidemiological SIR Model (Italy Example)
# ====================================
def sir_model(y, t, beta, gamma):
    S, I, R = y
    dSdt = -beta * S * I
    dIdt = beta * S * I - gamma * I
    dRdt = gamma * I
    return dSdt, dIdt, dRdt

def run_sir_model(country, population, beta=0.2, gamma=0.1):
    confirmed = owid[(owid['location'] == country) & (owid['date'] < '2021-06-01')]
    cases = confirmed['total_cases_per_million'].dropna().values / 1e6 * population
    if len(cases) < 2: return None
    I0 = cases[0]
    S0 = population - I0
    R0 = 0
    y0 = S0, I0, R0
    t = np.linspace(0, len(cases)-1, len(cases))
    ret = odeint(sir_model, y0, t, args=(beta, gamma))
    S, I, R = ret.T
    return t, I, cases

# Italy SIR
t, I, actual = run_sir_model("Italy", 60360000)
plt.figure(figsize=(10,5))
plt.plot(t, I, label="SIR Predicted Infected")
plt.plot(t, actual, label="Actual Cases")
plt.legend()
plt.title("🇮🇹 Italy - SIR Model vs Actual Cases")
plt.show()

# ====================================
# Summary
# ====================================
print("""
🔬 Project Summary:
- Vaccination shows strong negative correlation with deaths per million (confirmed by regression & statistical tests).
- GHSI (preparedness) alone is not sufficient — timing & public response matter.
- ML Models: Linear Regression performed slightly better than KNN in this context.
- SIR model shows potential to approximate infection spread patterns but depends heavily on parameter tuning.

🎓 This notebook demonstrates real-world public health insights using EDA, hypothesis testing, machine learning, and epidemiological modeling.
""")


URLError: <urlopen error [Errno -2] Name or service not known>

In [ ]:
# ====================================
# Step 1: Import Libraries
# ====================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr, ttest_ind

sns.set(style="whitegrid", palette="muted")

# ====================================
# Step 2: Load Global COVID-19 + GHSI Data
# ====================================
owid_url = 'https://covid.ourworldindata.org/data/owid-covid-data.csv'
owid = pd.read_csv(owid_url, parse_dates=['date'])

ox_url = 'https://pandemicdatalake.blob.core.windows.net/public/curated/covid-19/covid_policy_tracker/latest/covid_policy_tracker.csv'
policy = pd.read_csv(ox_url, parse_dates=['date'])  # Not used much as OWID has it included

ghsi = pd.read_csv('ghsi.csv')  # You must have this file locally
ghsi.rename(columns={'Country': 'location', 'Overall Score': 'ghsi_score'}, inplace=True)

# Select core columns from OWID
owid_data = owid[['location', 'date', 'total_cases_per_million', 'total_deaths_per_million',
                  'people_vaccinated_per_hundred', 'stringency_index']]

# ====================================
# Step 3: Create Snapshot for Global Comparison (fixed date)
# ====================================
snapshot_date = '2021-12-31'
snapshot = owid_data[owid_data['date'] == pd.to_datetime(snapshot_date)].dropna()

summary = snapshot[['location', 'total_cases_per_million', 'total_deaths_per_million',
                    'people_vaccinated_per_hundred', 'stringency_index']]
summary = summary.merge(ghsi[['location', 'ghsi_score']], on='location', how='left')
summary.dropna(inplace=True)

print("✔ Countries included:", summary.shape[0])
display(summary.describe())

# ====================================
# Step 4: Rich Exploratory Data Analysis (EDA)
# ====================================

# Pairwise plot
sns.pairplot(summary[['total_cases_per_million', 'total_deaths_per_million',
                      'people_vaccinated_per_hundred', 'stringency_index', 'ghsi_score']])
plt.suptitle("📈 Pairwise Comparison of Indicators", y=1.02)
plt.show()

# Correlation matrix heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(summary.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("🔁 Correlation Matrix")
plt.show()

# Histogram of key indicators
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
cols = ['people_vaccinated_per_hundred', 'total_deaths_per_million',
        'stringency_index', 'ghsi_score']
for i, col in enumerate(cols):
    sns.histplot(summary[col], kde=True, ax=axs[i // 2][i % 2])
    axs[i // 2][i % 2].set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

# Boxplot to show variation
plt.figure(figsize=(10, 5))
sns.boxplot(data=summary[['people_vaccinated_per_hundred', 'ghsi_score', 'stringency_index']])
plt.title("📦 Boxplots of Key Indicators")
plt.show()

# ====================================
# Step 5: Statistical Tests
# ====================================

# Pearson vs Spearman correlation
print("\n📊 Pearson/Spearman Correlations:")
pearson_r, _ = pearsonr(summary['people_vaccinated_per_hundred'], summary['total_deaths_per_million'])
spearman_r, _ = spearmanr(summary['people_vaccinated_per_hundred'], summary['total_deaths_per_million'])
print(f"Pearson: {pearson_r:.3f}")
print(f"Spearman: {spearman_r:.3f}")

# High vs Low Vaccination Countries (T-Test)
median_vax = summary['people_vaccinated_per_hundred'].median()
high_vax = summary[summary['people_vaccinated_per_hundred'] > median_vax]['total_deaths_per_million']
low_vax = summary[summary['people_vaccinated_per_hundred'] <= median_vax]['total_deaths_per_million']
t_stat, p_val = ttest_ind(high_vax, low_vax)

print(f"\n🧪 T-Test Between High and Low Vaccinated Countries:")
print(f"T-statistic = {t_stat:.2f}, p-value = {p_val:.5f}")
if p_val < 0.05:
    print("→ Statistically significant difference in deaths per million.")
else:
    print("→ No significant difference found.")

# ====================================
# Step 6: Experimental Curiosity Checks
# ====================================

# 👀 Experiment: Does higher GHSI always mean lower deaths?
plt.figure(figsize=(10, 6))
sns.scatterplot(data=summary, x='ghsi_score', y='total_deaths_per_million', hue='people_vaccinated_per_hundred', palette='viridis')
plt.title("🤔 Do Higher Preparedness Scores Mean Lower Deaths?")
plt.xlabel("GHSI Score")
plt.ylabel("COVID-19 Deaths per Million")
plt.show()

# 🌍 Region-level: Filter European Countries Only
europe = owid[owid['continent'] == 'Europe']
euro_snapshot = europe[europe['date'] == pd.to_datetime(snapshot_date)][
    ['location', 'people_vaccinated_per_hundred', 'total_deaths_per_million']
].dropna()

plt.figure(figsize=(12, 6))
sns.barplot(data=euro_snapshot.sort_values('people_vaccinated_per_hundred', ascending=False),
            x='location', y='people_vaccinated_per_hundred')
plt.xticks(rotation=90)
plt.title("🇪🇺 European Countries by Vaccination Rate")
plt.show()

# ====================================
# Step 7: Regression Model
# ====================================
X = summary[['people_vaccinated_per_hundred', 'stringency_index', 'ghsi_score']]
y = summary['total_deaths_per_million']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()

print("\n📉 Regression Summary:\n")
print(model.summary())

# ====================================
# Step 8: Time Series Analysis
# ====================================
countries = ['United States', 'India', 'Brazil', 'Germany', 'South Africa']
ts = owid_data[owid_data['location'].isin(countries)].set_index('date')

plt.figure(figsize=(14, 5))
for country in countries:
    df = ts[ts['location'] == country]
    plt.plot(df.index, df['people_vaccinated_per_hundred'], label=country)
plt.title("📆 Vaccination Uptake Over Time")
plt.xlabel("Date")
plt.ylabel("Vaccinated per 100 People")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
for country in countries:
    df = ts[ts['location'] == country]
    plt.plot(df.index, df['stringency_index'], label=country)
plt.title("🛡️ Stringency Index Over Time")
plt.xlabel("Date")
plt.ylabel("Stringency Index")
plt.legend()
plt.tight_layout()
plt.show()

# ====================================
# Final Reflection
# ====================================
print("""
🔍 Final Insights:
- Countries with higher vaccination rates had significantly lower death rates per million.
- Preparedness scores (GHSI) are not always predictive of better outcomes—likely due to policy delays or societal compliance.
- Regression shows vaccination and stringency index significantly explain death variation across nations.
- Time-series analysis shows timing of vaccine rollouts and policy differences can heavily affect outcomes.
""")


URLError: <urlopen error [Errno -2] Name or service not known>

In [ ]:
# ====================================
# Step 0: Import Libraries
# ====================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor # Corrected import
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from scipy.integrate import odeint
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

# ====================================
# Step 1: Load OWID COVID‑19 Dataset
# ====================================
try:
    owid = pd.read_csv(
        "owid-covid-data.csv", parse_dates=["date"]
    )
    print("✅ OWID data loaded.")
except FileNotFoundError:
    print("⚠️ Please download 'owid-covid-data.csv' from OWID and place it here.")
    raise

# ====================================
# Step 2: Load GHS Index CSV (2019 or 2021)
# ====================================
try:
    ghsi = pd.read_csv("ghsi_index.csv")  # downloaded from ghsindex.org archive :contentReference[oaicite:6]{index=6}
    ghsi.rename(columns={"Country": "location", "Overall Score": "ghsi_score"}, inplace=True)
except FileNotFoundError:
    print("⚠️ Please download a GHS Index CSV (2019 or 2021) and name it 'ghsi_index.csv'")
    raise

# ====================================
# Step 3: Filter and Prepare Data
# ====================================
# Select only the national-level data and impactful indicators
cols = [
    "location", "date", "total_cases_per_million", "total_deaths_per_million",
    "people_vaccinated_per_hundred", "stringency_index", "continent"
]
data = owid[cols].dropna(subset=["people_vaccinated_per_hundred"])

impactful = [
    "United Kingdom", "United States", "Germany", "India", "Brazil",
    "Israel", "South Africa", "Japan", "Russia", "Australia"
]
data = data[data["location"].isin(impactful)]

# ====================================
# Step 4: Snapshot (Dec 31 2021)
# ====================================
snapshot = data[data["date"] == pd.to_datetime("2021-12-31")]
summary = snapshot[[
    "location", "total_deaths_per_million", "people_vaccinated_per_hundred", "stringency_index"
]].copy()
summary = summary.merge(ghsi[["location", "ghsi_score"]], on="location", how="left").dropna()
summary = summary.sort_values(by="total_deaths_per_million", ascending=False)

print("📊 Snapshot Summary:")
print(summary)

# ====================================
# Step 5: Time-Series Visualization
# ====================================
plt.figure(figsize=(14, 6))
for c in impactful:
    dfc = data[data["location"] == c]
    plt.plot(dfc["date"], dfc["people_vaccinated_per_hundred"], label=c)
plt.title("Vaccination Progress – 10 Countries")
plt.xlabel("Date"); plt.ylabel("Vaccinated per 100"); plt.legend(); plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
for c in impactful:
    dfc = data[data["location"] == c]
    plt.plot(dfc["date"], dfc["total_deaths_per_million"], label=c)
plt.title("Deaths per Million Over Time")
plt.xlabel("Date"); plt.ylabel("Deaths per Million"); plt.legend(); plt.tight_layout()
plt.show()

# ====================================
# Step 6: Correlation and ML Models
# ====================================
corr = summary[[
    "total_deaths_per_million", "people_vaccinated_per_hundred", "stringency_index", "ghsi_score"
]].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix"); plt.tight_layout(); plt.show()

# Prepare ML features
X = summary[["people_vaccinated_per_hundred", "stringency_index", "ghsi_score"]]
y = summary["total_deaths_per_million"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression().fit(X_train, y_train)
knn = KNeighborsRegressor(n_neighbors=5).fit(X_train, y_train)
y_lr = lr.predict(X_test); y_knn = knn.predict(X_test)

print("Linear Regression R²:", r2_score(y_test, y_lr))
print("KNN Regression R²:", r2_score(y_test, y_knn))

plt.figure(figsize=(10, 5))
plt.plot(y_test.values, label="True Deaths")
plt.plot(y_lr, "--", label="Predicted LR")
plt.plot(y_knn, "--", label="Predicted KNN")
plt.legend(); plt.title("Model Predictions vs Actual"); plt.tight_layout(); plt.show()

# ====================================
# Step 7: Experimental Scatter Insights
# ====================================
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=summary,
    x="ghsi_score", y="total_deaths_per_million",
    size="people_vaccinated_per_hundred", hue="stringency_index",
    palette="viridis", sizes=(50, 300)
)
plt.title("Preparedness vs Mortality (Bubble = Vaccination)")
plt.xlabel("GHSI Score"); plt.ylabel("Deaths per Million"); plt.tight_layout(); plt.show()

# ====================================
# Step 8: SIR Model Spotlight – Best Country
# ====================================
# Choose country with clear vaccination/demise shift: e.g. Israel
def sir_model(y, t, beta, gamma):
    S, I, R = y
    return -beta * S * I, beta * S * I - gamma * I, gamma * I

def run_sir(country, pop, beta=0.3, gamma=0.1):
    dfc = data[data["location"] == country]
    dfc = dfc[dfc["total_cases_per_million"].notna()]
    cases = dfc["total_cases_per_million"].values / 1e6 * pop
    if len(cases) < 15:
        return None
    I0 = cases[0]; S0 = pop - I0; R0 = 0
    y0 = (S0, I0, R0)
    t = np.linspace(0, len(cases)-1, len(cases))
    ret = odeint(sir_model, y0, t, args=(beta, gamma))
    return t, ret[:,1], cases

# Spotlight Israel (fast rollout, clear drop)
pop_est = 9000000
res = run_sir("Israel", pop_est)
if res:
    t, I_pred, actual = res
    plt.figure(figsize=(10,5))
    plt.plot(t, I_pred, label="SIR Predicted Infected")
    plt.plot(t, actual, "--", label="Actual Cases")
    plt.title("🇮🇱 Israel SIR Model vs Actual Cases")
    plt.xlabel("Days Since First Case"); plt.ylabel("Number of Cases"); plt.legend(); plt.tight_layout(); plt.show()

# Summary Narrative
print("""
🔎 Key Takeaways:
- Countries with higher vaccination and stringency generally saw fewer deaths.
- GHSI (preparedness) showed mixed predictive power — not always lower mortality.
- Regression models showed vaccination as the most influential predictor globally.
- The SIR model for Israel demonstrates outbreak suppression following vaccine rollout.
""")

⚠️ Please download 'owid-covid-data.csv' from OWID and place it here.


FileNotFoundError: [Errno 2] No such file or directory: 'owid-covid-data.csv'

In [ ]:
# ====================================
# Step 1: Import Libraries
# ====================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr, ttest_ind

sns.set(style="whitegrid", palette="muted")

# ====================================
# Step 2: Load Global COVID-19 + GHSI Data
# ====================================
owid_url = 'https://covid.ourworldindata.org/data/owid-covid-data.csv'
owid = pd.read_csv(owid_url, parse_dates=['date'])

ox_url = 'https://pandemicdatalake.blob.core.windows.net/public/curated/covid-19/covid_policy_tracker/latest/covid_policy_tracker.csv'
policy = pd.read_csv(ox_url, parse_dates=['date'])  # Not used much as OWID has it included

ghsi = pd.read_csv('ghsi.csv')  # You must have this file locally
ghsi.rename(columns={'Country': 'location', 'Overall Score': 'ghsi_score'}, inplace=True)

# Select core columns from OWID
owid_data = owid[['location', 'date', 'total_cases_per_million', 'total_deaths_per_million',
                  'people_vaccinated_per_hundred', 'stringency_index']]

# ====================================
# Step 3: Create Snapshot for Global Comparison (fixed date)
# ====================================
snapshot_date = '2021-12-31'
snapshot = owid_data[owid_data['date'] == pd.to_datetime(snapshot_date)].dropna()

summary = snapshot[['location', 'total_cases_per_million', 'total_deaths_per_million',
                    'people_vaccinated_per_hundred', 'stringency_index']]
summary = summary.merge(ghsi[['location', 'ghsi_score']], on='location', how='left')
summary.dropna(inplace=True)

print("✔ Countries included:", summary.shape[0])
display(summary.describe())

# ====================================
# Step 4: Rich Exploratory Data Analysis (EDA)
# ====================================

# Pairwise plot
sns.pairplot(summary[['total_cases_per_million', 'total_deaths_per_million',
                      'people_vaccinated_per_hundred', 'stringency_index', 'ghsi_score']])
plt.suptitle("📈 Pairwise Comparison of Indicators", y=1.02)
plt.show()

# Correlation matrix heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(summary.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("🔁 Correlation Matrix")
plt.show()

# Histogram of key indicators
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
cols = ['people_vaccinated_per_hundred', 'total_deaths_per_million',
        'stringency_index', 'ghsi_score']
for i, col in enumerate(cols):
    sns.histplot(summary[col], kde=True, ax=axs[i // 2][i % 2])
    axs[i // 2][i % 2].set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

# Boxplot to show variation
plt.figure(figsize=(10, 5))
sns.boxplot(data=summary[['people_vaccinated_per_hundred', 'ghsi_score', 'stringency_index']])
plt.title("📦 Boxplots of Key Indicators")
plt.show()

# ====================================
# Step 5: Statistical Tests
# ====================================

# Pearson vs Spearman correlation
print("\n📊 Pearson/Spearman Correlations:")
pearson_r, _ = pearsonr(summary['people_vaccinated_per_hundred'], summary['total_deaths_per_million'])
spearman_r, _ = spearmanr(summary['people_vaccinated_per_hundred'], summary['total_deaths_per_million'])
print(f"Pearson: {pearson_r:.3f}")
print(f"Spearman: {spearman_r:.3f}")

# High vs Low Vaccination Countries (T-Test)
median_vax = summary['people_vaccinated_per_hundred'].median()
high_vax = summary[summary['people_vaccinated_per_hundred'] > median_vax]['total_deaths_per_million']
low_vax = summary[summary['people_vaccinated_per_hundred'] <= median_vax]['total_deaths_per_million']
t_stat, p_val = ttest_ind(high_vax, low_vax)

print(f"\n🧪 T-Test Between High and Low Vaccinated Countries:")
print(f"T-statistic = {t_stat:.2f}, p-value = {p_val:.5f}")
if p_val < 0.05:
    print("→ Statistically significant difference in deaths per million.")
else:
    print("→ No significant difference found.")

# ====================================
# Step 6: Experimental Curiosity Checks
# ====================================

# 👀 Experiment: Does higher GHSI always mean lower deaths?
plt.figure(figsize=(10, 6))
sns.scatterplot(data=summary, x='ghsi_score', y='total_deaths_per_million', hue='people_vaccinated_per_hundred', palette='viridis')
plt.title("🤔 Do Higher Preparedness Scores Mean Lower Deaths?")
plt.xlabel("GHSI Score")
plt.ylabel("COVID-19 Deaths per Million")
plt.show()

# 🌍 Region-level: Filter European Countries Only
europe = owid[owid['continent'] == 'Europe']
euro_snapshot = europe[europe['date'] == pd.to_datetime(snapshot_date)][
    ['location', 'people_vaccinated_per_hundred', 'total_deaths_per_million']
].dropna()

plt.figure(figsize=(12, 6))
sns.barplot(data=euro_snapshot.sort_values('people_vaccinated_per_hundred', ascending=False),
            x='location', y='people_vaccinated_per_hundred')
plt.xticks(rotation=90)
plt.title("🇪🇺 European Countries by Vaccination Rate")
plt.show()

# ====================================
# Step 7: Regression Model
# ====================================
X = summary[['people_vaccinated_per_hundred', 'stringency_index', 'ghsi_score']]
y = summary['total_deaths_per_million']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()

print("\n📉 Regression Summary:\n")
print(model.summary())

# ====================================
# Step 8: Time Series Analysis
# ====================================
countries = ['United States', 'India', 'Brazil', 'Germany', 'South Africa']
ts = owid_data[owid_data['location'].isin(countries)].set_index('date')

plt.figure(figsize=(14, 5))
for country in countries:
    df = ts[ts['location'] == country]
    plt.plot(df.index, df['people_vaccinated_per_hundred'], label=country)
plt.title("📆 Vaccination Uptake Over Time")
plt.xlabel("Date")
plt.ylabel("Vaccinated per 100 People")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
for country in countries:
    df = ts[ts['location'] == country]
    plt.plot(df.index, df['stringency_index'], label=country)
plt.title("🛡️ Stringency Index Over Time")
plt.xlabel("Date")
plt.ylabel("Stringency Index")
plt.legend()
plt.tight_layout()
plt.show()

# ====================================
# Final Reflection
# ====================================
print("""
🔍 Final Insights:
- Countries with higher vaccination rates had significantly lower death rates per million.
- Preparedness scores (GHSI) are not always predictive of better outcomes—likely due to policy delays or societal compliance.
- Regression shows vaccination and stringency index significantly explain death variation across nations.
- Time-series analysis shows timing of vaccine rollouts and policy differences can heavily affect outcomes.
""")


URLError: <urlopen error [Errno -2] Name or service not known>

In [ ]:
# Step 0: Import necessary libraries
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.integrate import odeint
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

# Step 1: Load JHU CSSE time series data from GitHub
confirmed_url = ("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/"
                 "master/csse_covid_19_data/csse_covid_19_time_series/"
                 "time_series_covid19_confirmed_global.csv")
deaths_url = ("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/"
              "master/csse_covid_19_data/csse_covid_19_time_series/"
              "time_series_covid19_deaths_global.csv")
vaccine_url = ("https://raw.githubusercontent.com/govex/COVID-19/master/"
               "data_tables/vaccine_data/global_data/"
               "time_series_covid19_vaccine_global.csv")

confirmed = pd.read_csv(confirmed_url)
deaths = pd.read_csv(deaths_url)
vaccine = pd.read_csv(vaccine_url)

# Step 2: Transform time-series to long format
def ts_to_long(df, value_name):
    df2 = df.drop(columns=["Lat","Long","Province/State", "UID"], errors='ignore') # Added 'UID' to columns to drop
    df_long = df2.melt(id_vars=["Country/Region"], var_name="date", value_name=value_name)
    df_long['date'] = pd.to_datetime(df_long['date'], errors='coerce') # Use errors='coerce' to handle parsing issues
    print(f"Columns after melt for {value_name}: {df_long.columns}") # Print columns after melt
    return df_long.groupby(['Country/Region','date']).sum().reset_index()

conf_long = ts_to_long(confirmed, "confirmed")
death_long = ts_to_long(deaths, "deaths")
# Rename 'Date' column to 'date' in vaccine DataFrame before processing and remove unnecessary rename
vaccine.rename(columns={"Date":"date"}, inplace=True)
vac_long = ts_to_long(vaccine.rename(columns={"Country_Region":"Country/Region"}), "vaccine_doses").dropna(subset=['date']) # Drop rows where date parsing failed

# Merge all three into a single DataFrame
df_all = conf_long.merge(death_long, on=["Country/Region","date"])
print(f"Shape after merging confirmed and deaths: {df_all.shape}") # Print shape after first merge
df_all = df_all.merge(vac_long, on=["Country/Region","date"])
print(f"Shape after merging with vaccine: {df_all.shape}") # Print shape after second merge


# Step 3: Normalize per million and vaccinations per hundred people
# For simplicity, assume population external dataset or use OWID later for vaccine percentage
pop = {c:1e6 for c in df_all["Country/Region"].unique()}  # placeholder, replace with actual population
df_all['cases_per_million'] = df_all['confirmed'] / pop[df_all['Country/Region'][0]] * 1e6
df_all['deaths_per_million'] = df_all['deaths'] / pop[df_all['Country/Region'][0]] * 1e6
df_all['vaxx_per_hundred'] = df_all['vaccine_doses'] / pop[df_all['Country/Region'][0]] * 100

# Step 4: Select impactful countries
countries = ["United Kingdom","United States","Germany","India","Brazil",
             "Israel","South Africa","Japan","Russia","Australia"]
df_sel = df_all[df_all["Country/Region"].isin(countries)].copy()

# Create snapshot date
cutoff = pd.to_datetime("2021-12-31")
snap = df_sel[df_sel['date']==cutoff]

summary = snap[['Country/Region','deaths_per_million','vaxx_per_hundred']].copy().rename(columns={'Country/Region':'country'})
summary.sort_values('deaths_per_million', ascending=False, inplace=True)

print("Global snapshot summary (Dec 31, 2021):")
print(summary)

# Step 5: Plot time-series of vaccination and deaths
plt.figure(figsize=(14,6))
for c in countries:
    sub = df_sel[df_sel["Country/Region"]==c]
    plt.plot(sub['date'], sub['vaxx_per_hundred'], label=c)
plt.title("Vaccination Doses Administered per Hundred (est.)")
plt.xlabel("Date"); plt.ylabel("Vaccine doses per 100"); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(14,6))
for c in countries:
    sub = df_sel[df_sel["Country/Region"]==c]
    plt.plot(sub['date'], sub['deaths_per_million'], label=c)
plt.title("Deaths per Million Over Time")
plt.xlabel("Date"); plt.ylabel("Deaths per Million"); plt.legend(); plt.tight_layout(); plt.show()

# Step 6: Correlation and ML modeling
corr = summary[['deaths_per_million','vaxx_per_hundred']].corr()
print("Correlation between vaccination (per 100) and deaths per million:\n", corr)

X = summary[['vaxx_per_hundred']]
y = summary['deaths_per_million']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

lr = LinearRegression().fit(X_train, y_train)
knn = KNeighborsRegressor(n_neighbors=3).fit(X_train, y_test)
print("Linear Regression R²:", r2_score(y_test, lr.predict(X_test)))
print("KNN R²:", r2_score(y_test, knn.predict(X_test)))

plt.figure(figsize=(8,5))
plt.scatter(y_test, lr.predict(X_test), label='LR preds')
plt.scatter(y_test, knn.predict(X_test), label='KNN preds')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel("True deaths per million"); plt.ylabel("Predicted"); plt.legend(); plt.title("Model Predictions vs Actual"); plt.tight_layout(); plt.show()

# Step 7: Experimental scatter-bubble plot
plt.figure(figsize=(8,6))
plt.scatter(summary['vaxx_per_hundred'], summary['deaths_per_million'], s=100)
for i,row in summary.iterrows():
    plt.text(row['vaxx_per_hundred'], row['deaths_per_million'], row['country'], fontsize=9)
plt.title("Vaccination vs Mortality (2021-12-31)")
plt.xlabel("Vaccine Doses per 100"); plt.ylabel("Deaths per Million"); plt.tight_layout(); plt.show()

# Step 8: SIR model spotlight - UK
def sir(y,t,beta,gamma): return (-beta*y[0]*y[1], beta*y[0]*y[1]-gamma*y[1], gamma*y[1])
def run_sir(country,pop,beta=0.2,gamma=0.1):
    sub = df_sel[df_sel['Country/Region']==country]
    cases = (sub['confirmed']/pop).values
    if len(cases)<15: return None
    I0 = cases[0]; S0 = pop - I0; R0 = 0; t = np.arange(len(cases))
    y0=(S0,I0,R0)
    ret=odeint(lambda y,t: sir(y,t,beta,gamma), y0, t)
    return t, ret[:,1], cases

pop_est=68e6
res = run_sir("United Kingdom", pop_est)
if res:
    t, I_pred, I_actual = res
    plt.figure(figsize=(10,5))
    plt.plot(t, I_pred, label="SIR Infected Est.")
    plt.plot(t, I_actual, '--', label="Actual Cases (estimated per population)")
    plt.title("🇬🇧 SIR Model vs Actual for UK")
    plt.xlabel("Days"); plt.ylabel("Proportion Infected"); plt.legend(); plt.tight_layout(); plt.show()

print("✅ Analysis complete using JHU CSSE and govex vaccine data via GitHub sources.")

Columns after melt for confirmed: Index(['Country/Region', 'date', 'confirmed'], dtype='object')
Columns after melt for deaths: Index(['Country/Region', 'date', 'deaths'], dtype='object')
Columns after melt for vaccine_doses: Index(['Country/Region', 'date', 'vaccine_doses'], dtype='object')
Shape after merging confirmed and deaths: (229743, 4)
Shape after merging with vaccine: (0, 5)


KeyError: 0